In [ ]:
library(terra)
library(tidyverse)
library(sf)

# 1. Setup
netcdf_dir <- "dataForRScripts/NetCDF_Chunks"
region <- "JoshuaTree"
variable <- "T_Max"

# 2. List Files
files <- list.files(netcdf_dir, pattern = paste0(region, "_", variable), full.names = TRUE)

# 3. Load as a Virtual Raster Cube (Instant)
r <- rast(files)

# 4. Define Time Indices
# Extract time from layer names or metadata (terra handles this automatically usually)
times <- time(r)
years <- as.numeric(format(times, "%Y"))

# 5. Calculate Climatologies (Vectorized & Fast)
print("Calculating Baseline Mean...")
baseline_indices <- which(years >= 1970 & years <= 2014)
baseline_rast <- subset(r, baseline_indices)
baseline_mean <- app(baseline_rast, mean, na.rm=TRUE) # Pixel-wise mean

print("Calculating Future Mean...")
future_indices <- which(years >= 2070 & years <= 2099)
future_rast <- subset(r, future_indices)
future_mean <- app(future_rast, mean, na.rm=TRUE)

# 6. Anomaly
anomaly <- future_mean - baseline_mean

# 7. Convert to Data Frame for ggplot (Only convert the final result!)
df_anomaly <- as.data.frame(anomaly, xy = TRUE, na.rm = TRUE)
colnames(df_anomaly)[3] <- "Anomaly_Value"

# 8. Plot
ggplot(df_anomaly, aes(x=x, y=y, fill=Anomaly_Value)) +
  geom_raster() +
  scale_fill_viridis_c(option = "magma") +
  coord_fixed() +
  labs(title = paste(variable, "Anomaly (2070-2099 vs Baseline)"))